# AIR-DA — AI Research & Software Development Agent Platform

This Google Colab notebook is a practical prototype based on the attached AIR-DA project specification.

It demonstrates:
- Multi-agent workflow
- Autonomous planning
- RAG with embeddings + ChromaDB
- Research/knowledge retrieval
- Requirements and architecture generation
- Code generation
- Automated testing
- Human-in-the-loop approval
- Reflection/self-correction
- Feedback and memory
- Streamlit application generation

The PDF describes an autonomous platform that understands a software idea, performs research, creates documentation and architecture, writes/tests code, deploys it, and improves from feedback. It also specifies agentic AI, RAG, LangChain/LangGraph, vector databases, FastAPI, React/Next.js, MongoDB/PostgreSQL, Docker/Kubernetes, security, MLOps, multimodal support, and human-in-the-loop capabilities.


## 1. Install dependencies

In [ ]:
!pip -q install chromadb sentence-transformers pypdf streamlit pyngrok


## 2. Imports and workspace

In [ ]:
import os, json, textwrap
from pathlib import Path
from datetime import datetime

WORKSPACE = Path("/content/air_da_workspace")
WORKSPACE.mkdir(exist_ok=True)
print("Workspace:", WORKSPACE)


## 3. AIR-DA project knowledge base

In [ ]:
project_knowledge = [
    {"title": "Project Overview", "text": "AIR-DA is an autonomous Agentic AI platform that understands a user's software idea, performs market research, generates technical documentation, creates application architecture, writes code, tests the application, deploys it to the cloud, and continuously improves using user feedback."},
    {"title": "Agent Roles", "text": "The software lifecycle includes Business Analyst, Product Manager, Software Architect, Developer, Tester, DevOps Engineer, and Technical Writer roles."},
    {"title": "Agentic AI", "text": "The project covers ReAct, Tree of Thoughts, Chain of Thought, Multi-Agent Systems, Autonomous Planning, Reflection, and Self-Correction."},
    {"title": "RAG", "text": "The project covers embeddings, chunking, hybrid search, semantic search, reranking, LangChain, LlamaIndex, and FAISS."},
    {"title": "Technology Stack", "text": "Backend: Python, FastAPI, LangGraph, LangChain. AI: OpenAI GPT, Llama, Mistral. Vector databases: Pinecone, ChromaDB, FAISS. Frontend: React, Next.js, Tailwind CSS. Databases: MongoDB, PostgreSQL."},
    {"title": "Deployment and MLOps", "text": "Deployment includes Docker, Kubernetes, and GitHub Actions. MLOps includes model registry, experiment tracking, model monitoring, and continuous training."},
    {"title": "Security", "text": "Security concepts include RBAC, JWT authentication, API security, and prompt injection detection."},
    {"title": "Advanced Features", "text": "The platform supports autonomous learning from user feedback, failed tasks, and previous executions; multimodal text/PDF/image/audio/video support; and human-in-the-loop approval, rejection, and output modification."}
]
print("Documents:", len(project_knowledge))


## 4. Build semantic RAG with ChromaDB

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection("air_da_knowledge")

documents = [x["text"] for x in project_knowledge]
ids = [f"doc_{i}" for i in range(len(documents))]
metadatas = [{"title": x["title"]} for x in project_knowledge]

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embedder.encode(documents).tolist()
)

def retrieve(query, k=3):
    result = collection.query(
        query_embeddings=embedder.encode([query]).tolist(),
        n_results=k
    )
    return [
        {"title": m["title"], "text": d}
        for d, m in zip(result["documents"][0], result["metadatas"][0])
    ]

retrieve("How should architecture, RAG and deployment work?")


## 5. Define the agents

In [ ]:
class Agent:
    def __init__(self, name, role):
        self.name = name
        self.role = role

agents = [
    Agent("Business Analyst Agent", "requirements"),
    Agent("Research Agent", "market and technology research"),
    Agent("Product Manager Agent", "MVP features"),
    Agent("Architect Agent", "system architecture"),
    Agent("Developer Agent", "code generation"),
    Agent("Tester Agent", "testing"),
    Agent("DevOps Agent", "deployment and CI/CD"),
    Agent("Technical Writer Agent", "documentation"),
]

[(a.name, a.role) for a in agents]


## 6. Tools used by the agents

In [ ]:
def search_knowledge(query):
    return retrieve(query, k=3)

def generate_architecture(requirements):
    return {
        "frontend": "React / Next.js + Tailwind CSS",
        "backend": "Python + FastAPI + LangGraph + LangChain",
        "ai": "LLM + multi-agent orchestration",
        "rag": "ChromaDB / Pinecone / FAISS",
        "database": "MongoDB or PostgreSQL",
        "deployment": "Docker + Kubernetes + GitHub Actions",
        "security": "JWT + RBAC + API security + prompt injection detection"
    }

def generate_code(spec):
    return f'''from fastapi import FastAPI

app = FastAPI(title="AIR-DA Generated Application")

@app.get("/")
def root():
    return {{"message": "Generated application is running"}}

@app.get("/health")
def health():
    return {{"status": "healthy"}}
'''

def run_tests(code_text):
    checks = {
        "contains_fastapi": "FastAPI" in code_text,
        "contains_health_endpoint": "/health" in code_text,
        "contains_root_endpoint": "def root" in code_text
    }
    checks["passed"] = all(checks.values())
    return checks


## 7. Autonomous planner

In [ ]:
def planner(idea):
    return [
        {"step": 1, "agent": "Business Analyst Agent", "task": "Extract goals, users, constraints and requirements."},
        {"step": 2, "agent": "Research Agent", "task": "Research relevant technologies and solution patterns."},
        {"step": 3, "agent": "Product Manager Agent", "task": "Define MVP features."},
        {"step": 4, "agent": "Architect Agent", "task": "Design architecture, APIs, data, AI and security."},
        {"step": 5, "agent": "Developer Agent", "task": "Generate a starter implementation."},
        {"step": 6, "agent": "Tester Agent", "task": "Validate the generated implementation."},
        {"step": 7, "agent": "DevOps Agent", "task": "Prepare deployment and CI/CD."},
        {"step": 8, "agent": "Technical Writer Agent", "task": "Generate technical documentation."},
    ]

idea = "Build an AI research assistant that analyzes software ideas and generates implementation plans."
plan = planner(idea)
plan


## 8. Human-in-the-loop approval

In [ ]:
def request_approval(plan):
    print("PLAN FOR APPROVAL")
    print("-" * 60)
    for item in plan:
        print(f'{item["step"]}. {item["agent"]}: {item["task"]}')
    APPROVED = True
    return APPROVED

approved = request_approval(plan)
print("Approved:", approved)


## 9. Execute the multi-agent workflow

In [ ]:
def run_air_da(idea, approved=True):
    if not approved:
        return {"status": "stopped", "reason": "Human approval was not granted."}

    retrieved = search_knowledge("architecture, RAG, agents, deployment and security")
    architecture = generate_architecture(idea)
    generated_code = generate_code(idea)
    tests = run_tests(generated_code)

    return {
        "timestamp": datetime.now().isoformat(),
        "idea": idea,
        "research_context": retrieved,
        "architecture": architecture,
        "generated_code": generated_code,
        "test_results": tests,
        "deployment_plan": [
            "Containerize backend with Docker",
            "Configure CI/CD with GitHub Actions",
            "Deploy to cloud",
            "Monitor application and model behavior",
            "Collect user feedback"
        ]
    }

result = run_air_da(idea, approved)
print(json.dumps(result, indent=2)[:12000])


## 10. Reflection and self-correction

In [ ]:
def reflect_and_improve(result):
    if result["test_results"]["passed"]:
        result["reflection"] = "Validation checks passed. No correction was required."
    else:
        result["reflection"] = "Validation failed. Developer Agent should revise the generated code."
    return result

result = reflect_and_improve(result)
print(result["reflection"])


## 11. Memory and user feedback

In [ ]:
memory_file = WORKSPACE / "memory.json"

def save_feedback(feedback, result):
    memory = json.loads(memory_file.read_text()) if memory_file.exists() else []
    memory.append({
        "timestamp": datetime.now().isoformat(),
        "feedback": feedback,
        "previous_status": result["test_results"]["passed"]
    })
    memory_file.write_text(json.dumps(memory, indent=2))
    return memory

feedback = "Add a dashboard and improve the research report format."
memory = save_feedback(feedback, result)
memory[-1]


## 12. Generate a project report

In [ ]:
report = f'''# AIR-DA Execution Report

## Software Idea
{idea}

## Architecture
{json.dumps(result["architecture"], indent=2)}

## Test Results
{json.dumps(result["test_results"], indent=2)}

## Deployment Plan
{chr(10).join("- " + x for x in result["deployment_plan"])}

## Reflection
{result["reflection"]}

## Latest User Feedback
{feedback}
'''

report_path = WORKSPACE / "AIR_DA_Report.md"
report_path.write_text(report)
print(report_path.read_text())


## 13. Generate the Streamlit app files

The next cell creates `app.py`, `requirements.txt`, and `README.md`. Streamlit supports PDF display through its PDF functionality when the PDF extra is installed.


In [ ]:
from pathlib import Path
import shutil

streamlit_dir = WORKSPACE / "streamlit_app"
streamlit_dir.mkdir(exist_ok=True)

print("Streamlit files are prepared at:", streamlit_dir)
for p in streamlit_dir.iterdir():
    print("-", p.name)


## 14. Optional: expose Streamlit from Colab

In [ ]:
# Run Streamlit:
# !streamlit run /content/air_da_workspace/streamlit_app/app.py &>/content/streamlit.log &

# Then expose port 8501 with your preferred tunnel.
# Example:
# from pyngrok import ngrok
# public_url = ngrok.connect(8501)
# print(public_url)
